# LeetCode 1810 — Minimum Path Cost in a Hidden Grid bằng Python

Link đề: https://leetcode.com/problems/minimum-path-cost-in-a-hidden-grid/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Heap (Priority Queue), Interactive, Matrix, Shortest Path

> 🔒 **Bài Premium, dạng tương tác (interactive).** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc. Vì không có máy chấm, notebook **tự dựng lớp giả lập `GridMaster`** đúng API `canMove` / `move` / `isTarget`.

Skeleton LeetCode:

```python
# class GridMaster(object):
#     def canMove(self, direction: str) -> bool: ...
#     def move(self, direction: str) -> int: ...
#     def isTarget(self) -> bool: ...

class Solution(object):
    def findShortestPath(self, master: 'GridMaster') -> int:
        ...
```

Lộ trình: bài tách làm **hai pha** — (A) **khám phá bản đồ** bằng DFS có quay lui qua API; (B) **tìm đường rẻ nhất** trên bản đồ đã biết. Pha B đi từ **vét cạn mọi đường đơn** → **Bellman–Ford (thư giãn lặp)** → **Dijkstra với heap** (lời giải nộp) → **Dial (hàng đợi theo xô, vì chi phí ≤ 100)**.

## 1. Mục tiêu học

1. Làm quen bài **tương tác**: không được nhìn dữ liệu, chỉ được "hỏi" qua API; tự dựng **lớp giả lập** để chạy thử và đếm số lần gọi.
2. Khám phá một mê cung chưa biết bằng **DFS có quay lui vật lý** (đi tới rồi phải **đi ngược lại**), dùng **tọa độ tương đối** so với điểm xuất phát.
3. Viết DFS khám phá **không đệ quy** (stack tường minh) để tránh giới hạn đệ quy với lưới `100 × 100`.
4. Giải bài **đường đi chi phí nhỏ nhất trên lưới có trọng số ở đỉnh** bằng **Dijkstra**, hiểu vì sao BFS thường không đủ.
5. Kiểm thử bằng "**oracle**" (Dijkstra trên lưới đầy đủ) trên lưới ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Có một robot trong một lưới **ẩn** kích thước `m × n`; mỗi ô **trống** hoặc **bị chặn**. Ô xuất phát và ô đích khác nhau và đều không bị chặn. Mỗi ô có **chi phí** phải trả **mỗi lần đi vào** ô đó; chi phí ô xuất phát **không** bị tính lúc đầu.

Bạn **không biết** kích thước lưới, vị trí xuất phát hay vị trí đích. Bạn chỉ có thể hỏi đối tượng `GridMaster`:
- `canMove(direction)` → `True` nếu robot đi được theo hướng đó.
- `move(direction)` → đi theo hướng đó và trả về **chi phí ô mới**. Nếu đi vào ô bị chặn hoặc ra ngoài lưới thì lệnh bị bỏ qua, robot đứng yên và trả về `-1`.
- `isTarget()` → `True` nếu robot đang đứng ở ô đích.

`direction ∈ {'U','D','L','R'}` (lên, xuống, trái, phải). Trả về **tổng chi phí nhỏ nhất** để đưa robot từ ô xuất phát tới ô đích, hoặc `-1` nếu không có đường.

**Dữ liệu kiểm thử tùy chỉnh** (chỉ để dựng giả lập — code của bạn không được biết): lưới `grid` với `grid[i][j] == 0` là ô chặn, `grid[i][j] ≥ 1` là chi phí; `(r1, c1)` là ô xuất phát; `(r2, c2)` là ô đích.

**Ví dụ 1:** `grid = [[2,3],[1,1]], r1 = 0, c1 = 1, r2 = 1, c2 = 0` → `2` (xem chú thích bên dưới).

**Ví dụ 2:** `grid = [[0,3,1],[3,4,2],[1,2,0]], r1 = 2, c1 = 0, r2 = 0, c2 = 2` → `9`, đường `(2,0)→(2,1)→(1,1)→(1,2)→(0,2)` có chi phí `2 + 4 + 2 + 1 = 9`.

**Ví dụ 3:** `grid = [[1,0],[0,1]], r1 = 0, c1 = 0, r2 = 1, c2 = 1` → `-1` (không có đường).

**Chú thích Ví dụ 1:** từ `(0,1)` (chi phí 3, không tính) có hai đường tới `(1,0)`: qua `(0,0)` tốn `2 + 1 = 3`, qua `(1,1)` tốn `1 + 1 = 2`. Đáp án `2`. (Phần giải thích trong đề chỉ minh họa *một* chuỗi tương tác khám phá, không phải đường tối ưu.)

**Ràng buộc:** `1 ≤ m, n ≤ 100`, `0 ≤ grid[i][j] ≤ 100`.

In [1]:
import heapq
import random
import sys
import time
from collections import deque
from typing import Dict, List, Optional, Tuple

DIRS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
OPP = {"U": "D", "D": "U", "L": "R", "R": "L"}

class GridMaster:
    """Giả lập đúng API của LeetCode, kèm bộ đếm số lần gọi."""
    def __init__(self, grid, r1, c1, r2, c2):
        self._g = grid
        self._m, self._n = len(grid), len(grid[0])
        self._pos = (r1, c1)
        self._target = (r2, c2)
        self.calls = {"canMove": 0, "move": 0, "isTarget": 0}

    def _next(self, d):
        dr, dc = DIRS[d]
        r, c = self._pos[0] + dr, self._pos[1] + dc
        if 0 <= r < self._m and 0 <= c < self._n and self._g[r][c] > 0:
            return r, c
        return None

    def canMove(self, direction: str) -> bool:
        self.calls["canMove"] += 1
        return self._next(direction) is not None

    def move(self, direction: str) -> int:
        self.calls["move"] += 1
        nxt = self._next(direction)
        if nxt is None:
            return -1
        self._pos = nxt
        return self._g[nxt[0]][nxt[1]]

    def isTarget(self) -> bool:
        self.calls["isTarget"] += 1
        return self._pos == self._target

examples = [
    (([[2, 3], [1, 1]], 0, 1, 1, 0), 2),
    (([[0, 3, 1], [3, 4, 2], [1, 2, 0]], 2, 0, 0, 2), 9),
    (([[1, 0], [0, 1]], 0, 0, 1, 1), -1),
]
gm = GridMaster(*examples[0][0])
print("thử API:", gm.canMove("U"), gm.canMove("L"), gm.move("L"), gm.isTarget(), gm.move("D"), gm.isTarget(), gm.calls)

thử API: False True 2 False 1 True {'canMove': 2, 'move': 2, 'isTarget': 2}


## 3. Kiến thức nền

### 3.1 Tọa độ tương đối và `dict` làm bản đồ

Không biết vị trí thật của robot, ta coi ô xuất phát là `(0, 0)`; đi `U` thì hàng giảm 1, `D` tăng 1, `L` cột giảm 1, `R` tăng 1. Bản đồ là `dict` `{(r, c): chi_phí}` — tọa độ có thể **âm**, không cần biết kích thước.

In [2]:
pos = (0, 0)
cost_map = {pos: 0}
for d, c in [("L", 2), ("D", 1)]:
    dr, dc = DIRS[d]
    pos = (pos[0] + dr, pos[1] + dc)
    cost_map[pos] = c
print(cost_map)

{(0, 0): 0, (0, -1): 2, (1, -1): 1}


### 3.2 Quay lui **vật lý**

Trong DFS thường, "quay lui" chỉ là `return`. Ở đây robot **thật sự** đứng ở ô con, nên sau khi khám phá xong ô con phải gọi `move(OPP[d])` để đưa robot về ô cha — nếu không, vị trí ta **nghĩ** robot đang đứng sẽ lệch với vị trí **thật**.

In [3]:
gm = GridMaster([[1, 1, 1]], 0, 0, 0, 2)
print("đi R:", gm.move("R"), "| quay lại L:", gm.move("L"), "| vị trí nội bộ:", gm._pos)

đi R: 1 | quay lại L: 1 | vị trí nội bộ: (0, 0)


### 3.3 `heapq` cho Dijkstra

Hàng đợi ưu tiên lưu `(khoảng_cách, đỉnh)`; `heappop` lấy đỉnh có khoảng cách tạm nhỏ nhất. Bản "lazy": một đỉnh có thể bị đẩy nhiều lần, khi pop ra ta bỏ qua nếu khoảng cách đã lỗi thời.

In [4]:
pq = [(5, "a"), (2, "b"), (9, "c")]
heapq.heapify(pq)
print([heapq.heappop(pq) for _ in range(3)])

[(2, 'b'), (5, 'a'), (9, 'c')]


### 3.4 Trọng số ở **đỉnh** thay vì ở cạnh

Đi vào ô `v` tốn `cost[v]`. Có thể coi mọi cạnh `u → v` có trọng số `cost[v]` (đồ thị có hướng, trọng số dương) — Dijkstra áp dụng được vì mọi trọng số ≥ 1.

In [5]:
cost = {(0, 0): 0, (0, 1): 5, (1, 0): 1}
edges = [((0, 0), (0, 1)), ((0, 1), (0, 0)), ((0, 0), (1, 0))]
for u, v in edges:
    print(f"{u} -> {v}: trọng số = cost[v] = {cost[v]}")

(0, 0) -> (0, 1): trọng số = cost[v] = 5
(0, 1) -> (0, 0): trọng số = cost[v] = 0
(0, 0) -> (1, 0): trọng số = cost[v] = 1


### 3.5 Giới hạn đệ quy

Một lưới `100 × 100` dạng "rắn" có đường DFS dài ~5000 ô. DFS đệ quy sẽ vượt `sys.getrecursionlimit()` (mặc định 1000). Ta sẽ viết khám phá bằng **stack tường minh**.

In [6]:
print("recursion limit:", sys.getrecursionlimit())

recursion limit: 1000


## 4. Ý tưởng nền tảng / trực giác

Vì không nhìn thấy lưới, bài tách làm hai pha:

**Pha A — vẽ bản đồ.** Từ ô hiện tại, thử 4 hướng; hướng nào `canMove` và ô đó **chưa thăm** thì `move` vào (ghi lại chi phí trả về và `isTarget()`), khám phá tiếp đệ quy, rồi `move` hướng ngược lại để về. Kết thúc pha A ta có **toàn bộ thành phần liên thông** chứa ô xuất phát, kèm chi phí từng ô và vị trí đích (nếu tới được).

**Pha B — tìm đường rẻ nhất** trên bản đồ đã biết: bài toán đường đi ngắn nhất có trọng số dương ⇒ Dijkstra.

Tính tay Ví dụ 2 sau pha A (tọa độ tương đối, xuất phát `(2,0)` ↦ `(0,0)`):

```
tuyệt đối         tương đối (hàng-2, cột-0)
(1,0) cost 3  ->  (-1, 0) cost 3
(2,1) cost 2  ->  ( 0, 1) cost 2
(1,1) cost 4  ->  (-1, 1) cost 4
(0,1) cost 3  ->  (-2, 1) cost 3
(1,2) cost 2  ->  (-1, 2) cost 2
(0,2) cost 1  ->  (-2, 2) cost 1   <- đích
```

Đường rẻ nhất: `(0,0)→(0,1)→(-1,1)→(-1,2)→(-2,2)`: `2 + 4 + 2 + 1 = 9`; đường khác qua `(-1,0)→(-1,1)→(-2,1)→(-2,2)`: `3 + 4 + 3 + 1 = 11`.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Pha A: khám phá bằng DFS có quay lui (stack tường minh)

Mỗi phần tử stack là `[r, c, hướng_kế_tiếp_cần_thử, hướng_để_quay_về_cha]`.

In [7]:
ORDER = "URDL"

def explore(master):
    """Trả về (cost_map, target) với tọa độ tương đối; target = None nếu không gặp đích."""
    cost_map = {(0, 0): 0}
    target = (0, 0) if master.isTarget() else None
    stack = [[0, 0, 0, None]]
    while stack:
        top = stack[-1]
        r, c, k, back = top
        if k == 4:                       # đã thử đủ 4 hướng: quay về cha
            stack.pop()
            if back is not None:
                master.move(back)
            continue
        top[2] += 1
        d = ORDER[k]
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if (nr, nc) in cost_map:          # đã thăm
            continue
        if master.canMove(d):
            cost_map[(nr, nc)] = master.move(d)
            if master.isTarget():
                target = (nr, nc)
            stack.append([nr, nc, 0, OPP[d]])
    return cost_map, target

gm = GridMaster(*examples[1][0])
cm, tg = explore(gm)
print("bản đồ:", cm)
print("đích:", tg, "| số lần gọi API:", gm.calls, "| robot về đúng chỗ cũ:", gm._pos == (2, 0))

bản đồ: {(0, 0): 0, (-1, 0): 3, (-1, 1): 4, (-2, 1): 3, (-2, 2): 1, (-1, 2): 2, (0, 1): 2}
đích: (-2, 2) | số lần gọi API: {'canMove': 18, 'move': 12, 'isTarget': 7} | robot về đúng chỗ cũ: True


### 5.2 Pha B baseline: vét cạn mọi đường đơn

Liệt kê mọi đường **đơn** (không lặp ô) từ `(0,0)` tới đích trên bản đồ, lấy tổng chi phí nhỏ nhất. Đường tối ưu với trọng số dương luôn là đường đơn, nên vét cạn này **đúng** — nhưng số đường đơn tăng theo hàm mũ.

In [8]:
def cheapest_bruteforce(cost_map, target):
    if target is None:
        return -1
    best = [float("inf")]
    seen = {(0, 0)}
    def dfs(u, acc):
        if acc >= best[0]:
            return                          # cắt nhánh đơn giản
        if u == target:
            best[0] = acc; return
        for dr, dc in DIRS.values():
            v = (u[0] + dr, u[1] + dc)
            if v in cost_map and v not in seen:
                seen.add(v); dfs(v, acc + cost_map[v]); seen.remove(v)
    dfs((0, 0), 0)
    return best[0] if best[0] < float("inf") else -1

class SolutionBrute:
    def findShortestPath(self, master) -> int:
        return cheapest_bruteforce(*explore(master))

for args, want in examples:
    print(SolutionBrute().findShortestPath(GridMaster(*args)), "| kỳ vọng", want)

2 | kỳ vọng 2
9 | kỳ vọng 9
-1 | kỳ vọng -1


**Độ phức tạp:** pha A `O(V)` (mỗi ô vào/ra một lần, ≤ 4 `canMove` mỗi ô). Pha B vét cạn: số đường đơn trên lưới có thể **hàm mũ** — một lưới `6 × 6` trống đã có hàng triệu đường. **Nút thắt:** khám phá lại cùng một ô theo nhiều đường; cần ghi nhớ "chi phí tốt nhất tới mỗi ô".

In [9]:
open_grid = [[1] * 5 for _ in range(5)]
t = time.perf_counter()
r = SolutionBrute().findShortestPath(GridMaster(open_grid, 0, 0, 4, 4))
print(f"lưới 5x5 trống: {r} trong {time.perf_counter() - t:.2f}s (đã có cắt nhánh)")

lưới 5x5 trống: 8 trong 0.00s (đã có cắt nhánh)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — pha A chỉ cần làm một lần, tốn `O(V)` lệnh API.** Mỗi ô của thành phần liên thông được `move` vào đúng 1 lần và `move` ra đúng 1 lần (cạnh cây DFS), cộng tối đa 4 `canMove` và 1 `isTarget` ⇒ tổng `≤ 7V` lời gọi.

**Insight 2 — pha B là đường đi ngắn nhất với trọng số dương ⇒ Dijkstra.** *Nếu* lấy ra khỏi heap ô `u` với khoảng cách `d` nhỏ nhất *thì* `d` là chi phí tối ưu tới `u` (mọi đường khác phải đi qua một ô còn trong heap có khoảng cách ≥ `d`, rồi cộng thêm chi phí dương).

**Insight 3 — BFS không đủ:** BFS tìm đường **ít bước** nhất, không phải **rẻ** nhất (xem Lỗi 3).

**Insight 4 — chi phí nguyên nhỏ (≤ 100):** có thể thay heap bằng **xô** (Dial's algorithm): khoảng cách tối đa `≤ 100 · V`, duyệt xô theo thứ tự tăng.

**Invariant Dijkstra:** mọi ô đã "chốt" (pop ra lần đầu với khoảng cách đúng) có `dist` tối ưu; mọi ô trong heap có `dist` tạm là chi phí tốt nhất qua các ô đã chốt.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Pha B bằng Bellman–Ford / thư giãn lặp (`O(V · E)`)

Lặp: với mọi cạnh `u → v`, *nếu* `dist[u] + cost[v] < dist[v]` *thì* cập nhật. Dừng khi một vòng không cập nhật gì (tối đa `V − 1` vòng).

In [10]:
def cheapest_bellman(cost_map, target):
    if target is None:
        return -1
    INF = float("inf")
    dist = {u: INF for u in cost_map}
    dist[(0, 0)] = 0
    for _ in range(len(cost_map)):
        changed = False
        for u in cost_map:
            if dist[u] == INF:
                continue
            for dr, dc in DIRS.values():
                v = (u[0] + dr, u[1] + dc)
                if v in cost_map and dist[u] + cost_map[v] < dist[v]:
                    dist[v] = dist[u] + cost_map[v]; changed = True
        if not changed:
            break
    return dist[target]

class SolutionBellman:
    def findShortestPath(self, master) -> int:
        return cheapest_bellman(*explore(master))

for args, want in examples:
    print(SolutionBellman().findShortestPath(GridMaster(*args)), "| kỳ vọng", want)

2 | kỳ vọng 2
9 | kỳ vọng 9
-1 | kỳ vọng -1


**Độ phức tạp:** worst `O(V · E) = O(V²)` (với `E ≤ 4V`) ⇒ `10⁸` cho `V = 10⁴` — chậm nhưng đúng; thực tế số vòng thường nhỏ hơn nhiều `V`.

### 7.2 Cách 3 — Pha B bằng Dijkstra với heap (lời giải chuẩn, `O(V log V)`)

In [11]:
def cheapest_dijkstra(cost_map, target, verbose=False):
    if target is None:
        return -1
    dist = {(0, 0): 0}
    pq = [(0, (0, 0))]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue                          # bản lỗi thời
        if verbose:
            print(f"chốt {u} với chi phí {d}")
        if u == target:
            return d
        for dr, dc in DIRS.values():
            v = (u[0] + dr, u[1] + dc)
            if v in cost_map:
                nd = d + cost_map[v]
                if nd < dist.get(v, float("inf")):
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
    return -1

cm, tg = explore(GridMaster(*examples[1][0]))
print("=>", cheapest_dijkstra(cm, tg, verbose=True))

chốt (0, 0) với chi phí 0
chốt (0, 1) với chi phí 2
chốt (-1, 0) với chi phí 3
chốt (-1, 1) với chi phí 6
chốt (-1, 2) với chi phí 8
chốt (-2, 1) với chi phí 9
chốt (-2, 2) với chi phí 9
=> 9


**Độ phức tạp:** mỗi ô bị đẩy tối đa (số cạnh vào) ≤ 4 lần ⇒ `O(E log E) = O(V log V)`; bộ nhớ `O(V)`.

### 7.3 Cách 4 — Pha B bằng Dial (xô theo chi phí, `O(V · C_max)`)

Vì mọi chi phí là số nguyên trong `[1, 100]`, khoảng cách tối đa ≤ `100 · V`. Dùng mảng xô `buckets[d]` = các ô có khoảng cách tạm `d`, duyệt `d = 0, 1, 2, …`. Để không phải duyệt `10⁶` xô rỗng, dùng **xô vòng** kích thước `C_max + 1 = 101`.

In [12]:
def cheapest_dial(cost_map, target, cmax=100):
    if target is None:
        return -1
    INF = float("inf")
    dist = {(0, 0): 0}
    size = cmax + 1
    buckets = [[] for _ in range(size)]
    buckets[0].append((0, 0))
    pending = 1
    d = 0
    while pending:
        b = buckets[d % size]
        while b:
            u = b.pop(); pending -= 1
            if dist[u] != d:
                continue
            if u == target:
                return d
            for dr, dc in DIRS.values():
                v = (u[0] + dr, u[1] + dc)
                if v in cost_map:
                    nd = d + cost_map[v]
                    if nd < dist.get(v, INF):
                        dist[v] = nd
                        buckets[nd % size].append(v); pending += 1
        d += 1
    return -1

class SolutionDial:
    def findShortestPath(self, master) -> int:
        return cheapest_dial(*explore(master))

for args, want in examples:
    print(SolutionDial().findShortestPath(GridMaster(*args)), "| kỳ vọng", want)

2 | kỳ vọng 2
9 | kỳ vọng 9
-1 | kỳ vọng -1


Xô vòng đúng vì tại thời điểm xử lý khoảng cách `d`, mọi khoảng cách tạm đang chờ đều nằm trong `[d, d + C_max]` — không có hai khoảng cách khác nhau rơi cùng một xô vòng. **Độ phức tạp:** `O(V + E + D_max)` với `D_max ≤ 100 V`.

## 8. TRACE từng bước

### 8.1 Trace pha A trên Ví dụ 1

`grid = [[2,3],[1,1]]`, xuất phát `(0,1)` ↦ tương đối `(0,0)`; thứ tự thử `U, R, D, L`.

In [13]:
def explore_trace(master):
    cost_map = {(0, 0): 0}
    target = (0, 0) if master.isTarget() else None
    stack = [[0, 0, 0, None]]
    while stack:
        top = stack[-1]
        r, c, k, back = top
        if k == 4:
            stack.pop()
            if back is not None:
                master.move(back)
                print(f"  xong ({r},{c}) -> quay về bằng move('{back}')")
            continue
        top[2] += 1
        d = ORDER[k]
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if (nr, nc) in cost_map:
            print(f"  tại ({r},{c}) thử {d}: ({nr},{nc}) đã thăm"); continue
        if master.canMove(d):
            cst = master.move(d)
            cost_map[(nr, nc)] = cst
            tgt = master.isTarget()
            if tgt: target = (nr, nc)
            print(f"  tại ({r},{c}) thử {d}: move -> ({nr},{nc}) chi phí {cst}{'  <- ĐÍCH' if tgt else ''}")
            stack.append([nr, nc, 0, OPP[d]])
        else:
            print(f"  tại ({r},{c}) thử {d}: bị chặn")
    return cost_map, target

gm = GridMaster(*examples[0][0])
cm, tg = explore_trace(gm)
print("bản đồ:", cm, "| đích:", tg, "| API:", gm.calls)

  tại (0,0) thử U: bị chặn
  tại (0,0) thử R: bị chặn
  tại (0,0) thử D: move -> (1,0) chi phí 1
  tại (1,0) thử U: (0,0) đã thăm
  tại (1,0) thử R: bị chặn
  tại (1,0) thử D: bị chặn
  tại (1,0) thử L: move -> (1,-1) chi phí 1  <- ĐÍCH
  tại (1,-1) thử U: move -> (0,-1) chi phí 2
  tại (0,-1) thử U: bị chặn
  tại (0,-1) thử R: (0,0) đã thăm
  tại (0,-1) thử D: (1,-1) đã thăm
  tại (0,-1) thử L: bị chặn
  xong (0,-1) -> quay về bằng move('D')
  tại (1,-1) thử R: (1,0) đã thăm
  tại (1,-1) thử D: bị chặn
  tại (1,-1) thử L: bị chặn
  xong (1,-1) -> quay về bằng move('R')
  xong (1,0) -> quay về bằng move('U')
  tại (0,0) thử L: (0,-1) đã thăm
bản đồ: {(0, 0): 0, (1, 0): 1, (1, -1): 1, (0, -1): 2} | đích: (1, -1) | API: {'canMove': 11, 'move': 6, 'isTarget': 4}


### 8.2 Trace pha B (Dijkstra) trên bản đồ vừa vẽ

| bước | pop `(d, ô)` | cập nhật |
|---|---|---|
| 1 | `(0, (0,0))` | `(0,-1)`: 0+2 = 2; `(1,0)`: 0+1 = 1 |
| 2 | `(1, (1,0))` | `(1,-1)`: 1+1 = 2 |
| 3 | `(2, (0,-1))` hoặc `(2, (1,-1))` | … |
| – | pop `(2, (1,-1))` = đích | **trả 2** |

In [14]:
print("=>", cheapest_dijkstra(cm, tg, verbose=True))

chốt (0, 0) với chi phí 0
chốt (1, 0) với chi phí 1
chốt (0, -1) với chi phí 2
chốt (1, -1) với chi phí 2
=> 2


## 9. Code nộp LeetCode

LeetCode (Python 3) cung cấp sẵn `GridMaster`; ta chỉ nộp `Solution`. Khám phá viết bằng stack nên không lo giới hạn đệ quy.

In [15]:
import heapq

class Solution(object):
    def findShortestPath(self, master: 'GridMaster') -> int:
        moves = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
        back = {"U": "D", "D": "U", "L": "R", "R": "L"}
        order = "URDL"

        # Pha A: DFS khám phá, tọa độ tương đối với ô xuất phát (0, 0)
        cost = {(0, 0): 0}
        target = None
        stack = [[0, 0, 0, None]]            # r, c, hướng kế tiếp, hướng quay về cha
        while stack:
            top = stack[-1]
            r, c, k, ret = top
            if k == 4:
                stack.pop()
                if ret is not None:
                    master.move(ret)         # quay lui VẬT LÝ
                continue
            top[2] += 1
            d = order[k]
            nr, nc = r + moves[d][0], c + moves[d][1]
            if (nr, nc) in cost or not master.canMove(d):
                continue
            cost[(nr, nc)] = master.move(d)
            if master.isTarget():
                target = (nr, nc)
            stack.append([nr, nc, 0, back[d]])

        if target is None:
            return -1

        # Pha B: Dijkstra trên bản đồ đã biết
        dist = {(0, 0): 0}
        pq = [(0, 0, 0)]
        while pq:
            dcur, r, c = heapq.heappop(pq)
            if (r, c) == target:
                return dcur
            if dcur > dist[(r, c)]:
                continue
            for dr, dc in moves.values():
                v = (r + dr, c + dc)
                if v in cost:
                    nd = dcur + cost[v]
                    if nd < dist.get(v, float("inf")):
                        dist[v] = nd
                        heapq.heappush(pq, (nd, v[0], v[1]))
        return -1

for args, want in examples:
    assert Solution().findShortestPath(GridMaster(*args)) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Pha A khám phá đúng thành phần liên thông.** Bất biến: khi phần tử `[r, c, k, ret]` ở **đỉnh** stack, robot thật đang đứng ở ô tương đối `(r, c)`. *Đúng lúc đầu* (`(0,0)`). Khi đẩy ô con, ta vừa `move(d)` thành công nên robot ở ô con. Khi pop ô con, ta `move(ret)` với `ret = OPP[d]` — ô cha chắc chắn trống (robot vừa từ đó đi sang) nên lệnh thành công, robot về ô cha = đỉnh stack mới. Nhờ vậy `canMove`/`move` luôn được hỏi đúng vị trí, và DFS thăm **mọi** ô tới được từ ô xuất phát (lập luận DFS chuẩn). Chi phí ghi lại là giá trị `move` trả về — đúng chi phí ô đó. Đích được phát hiện nếu và chỉ nếu nó tới được.

**Pha B (Dijkstra) đúng** vì mọi trọng số (chi phí ô) ≥ 1 > 0. Chứng minh chuẩn: ô đầu tiên được pop với khoảng cách `d` mà chưa chốt có `d` tối ưu — nếu có đường rẻ hơn, đường đó phải rời tập đã chốt tại một ô `x` còn trong heap với `dist[x] ≤` chi phí đường đó `< d`, mâu thuẫn với việc `d` nhỏ nhất. Ô xuất phát có `dist = 0` (chi phí ô xuất phát không tính) — khớp đề.

Nếu đích không tới được, pha A không gặp đích ⇒ trả `-1`.

## 11. Phân tích Time / Space

Gọi `V` = số ô tới được (≤ `m·n ≤ 10⁴`), `E ≤ 4V` cạnh, `C ≤ 100`.

| Phần | Time | Space | Số lời gọi API |
|---|---|---|---|
| Pha A (DFS stack) | `O(V)` | `O(V)` | `≤ 4V` canMove, `2(V−1)` move, `V` isTarget |
| Pha B — vét cạn đường đơn | hàm mũ | `O(V)` | 0 |
| Pha B — Bellman–Ford | `O(V·E)` | `O(V)` | 0 |
| Pha B — Dijkstra | `O(E log V)` | `O(V)` | 0 |
| Pha B — Dial | `O(V + E + C·V)` | `O(V + C)` | 0 |

In [16]:
def random_grid(m, n, rng, p_open=0.7):
    return [[rng.randint(1, 100) if rng.random() < p_open else 0 for _ in range(n)] for _ in range(m)]

def oracle(grid, r1, c1, r2, c2):
    """Dijkstra trên lưới đầy đủ (biết hết) — đáp án chuẩn."""
    m, n = len(grid), len(grid[0])
    dist = {(r1, c1): 0}; pq = [(0, r1, c1)]
    while pq:
        d, r, c = heapq.heappop(pq)
        if (r, c) == (r2, c2): return d
        if d > dist[(r, c)]: continue
        for dr, dc in DIRS.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] > 0:
                nd = d + grid[nr][nc]
                if nd < dist.get((nr, nc), float("inf")):
                    dist[(nr, nc)] = nd; heapq.heappush(pq, (nd, nr, nc))
    return -1

rng = random.Random(1810)
g = random_grid(100, 100, rng, 0.75)
g[0][0] = g[99][99] = 5
for name, sol in [("Nộp (Dijkstra)", Solution()), ("Dial", SolutionDial()), ("Bellman", SolutionBellman())]:
    gm = GridMaster(g, 0, 0, 99, 99)
    t = time.perf_counter(); r = sol.findShortestPath(gm); dt = time.perf_counter() - t
    print(f"{name:15s}: {r} trong {1000*dt:.0f} ms | API: {gm.calls}")
print("oracle:", oracle(g, 0, 0, 99, 99))

Nộp (Dijkstra) : 7256 trong 22 ms | API: {'canMove': 14983, 'move': 14914, 'isTarget': 7457}
Dial           : 7256 trong 19 ms | API: {'canMove': 14983, 'move': 14914, 'isTarget': 7458}


Bellman        : 7256 trong 561 ms | API: {'canMove': 14983, 'move': 14914, 'isTarget': 7458}
oracle: 7256


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu oracle)

In [17]:
sols = [Solution(), SolutionBellman(), SolutionDial()]
edge = examples + [
    (([[1, 1]], 0, 0, 0, 1), 1),                         # hai ô kề nhau
    (([[1, 100, 1], [1, 1, 1]], 0, 0, 0, 2), 4),         # đường vòng rẻ hơn đường thẳng
    (([[5, 0, 1]], 0, 0, 0, 2), -1),                     # bị chặn ở giữa
    (([[1], [2], [3], [4]], 3, 0, 0, 0), 6),             # một cột, đi lên
    (([[7, 1, 1, 1, 1]], 0, 4, 0, 0), 10),               # đích có chi phí cao
]
for args, want in edge:
    assert oracle(*args) == want
    for s in sols:
        gm = GridMaster(*args)
        assert s.findShortestPath(gm) == want, (args, s)
    assert SolutionBrute().findShortestPath(GridMaster(*args)) == want
    print("OK", args[0], "->", want)

OK [[2, 3], [1, 1]] -> 2
OK [[0, 3, 1], [3, 4, 2], [1, 2, 0]] -> 9
OK [[1, 0], [0, 1]] -> -1
OK [[1, 1]] -> 1
OK [[1, 100, 1], [1, 1, 1]] -> 4
OK [[5, 0, 1]] -> -1
OK [[1], [2], [3], [4]] -> 6
OK [[7, 1, 1, 1, 1]] -> 10


### 12.2 Lưới ngẫu nhiên seed cố định: so với oracle và vét cạn

Lưới nhỏ (≤ `4 × 4`) để vét cạn đường đơn chạy nhanh; thêm lưới vừa `30 × 30` so với oracle. Kiểm luôn: sau pha A robot **về đúng ô xuất phát** và số lời gọi API nằm trong cận `7V`.

In [18]:
rng = random.Random(42)
for _ in range(400):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    if m * n < 2:
        continue
    g = random_grid(m, n, rng, rng.choice([0.5, 0.7, 0.9]))
    cells = [(i, j) for i in range(m) for j in range(n)]
    (r1, c1), (r2, c2) = rng.sample(cells, 2)
    g[r1][c1] = g[r1][c1] or rng.randint(1, 100)
    g[r2][c2] = g[r2][c2] or rng.randint(1, 100)
    want = oracle(g, r1, c1, r2, c2)
    for s in sols + [SolutionBrute()]:
        gm = GridMaster(g, r1, c1, r2, c2)
        assert s.findShortestPath(gm) == want, (g, r1, c1, r2, c2)
    gm = GridMaster(g, r1, c1, r2, c2)
    cmap, _ = explore(gm)
    assert gm._pos == (r1, c1) and sum(gm.calls.values()) <= 7 * len(cmap) + 1
for _ in range(20):
    g = random_grid(30, 30, rng, 0.7); g[0][0] = g[29][29] = 1
    want = oracle(g, 0, 0, 29, 29)
    assert all(s.findShortestPath(GridMaster(g, 0, 0, 29, 29)) == want for s in [Solution(), SolutionDial()])
print("400 lưới nhỏ (oracle + vét cạn) + 20 lưới 30x30 (oracle): khớp")

400 lưới nhỏ (oracle + vét cạn) + 20 lưới 30x30 (oracle): khớp


### 12.3 Lưới "rắn" dài — kiểm tra không vỡ đệ quy

In [19]:
def snake(m, n):
    g = [[0] * n for _ in range(m)]
    for i in range(0, m, 2):
        for j in range(n):
            g[i][j] = 1
        if i + 1 < m:
            g[i + 1][n - 1 if (i // 2) % 2 == 0 else 0] = 1
    return g

g = snake(99, 100)
gm = GridMaster(g, 0, 0, 98, 99 if (98 // 2) % 2 == 0 else 0)
r = Solution().findShortestPath(gm)
print("độ dài đường rắn (số ô - 1):", sum(map(sum, g)) - 1, "| đáp án:", r, "| oracle:", oracle(g, 0, 0, *gm._target))

độ dài đường rắn (số ô - 1): 5048 | đáp án: 5048 | oracle: 5048


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên quay lui vật lý sau khi khám phá ô con

In [20]:
def explore_no_backtrack(master):
    cost_map = {(0, 0): 0}; target = None
    stack = [[0, 0, 0]]
    while stack:
        top = stack[-1]; r, c, k = top
        if k == 4:
            stack.pop(); continue                       # SAI: robot vẫn đứng ở ô con
        top[2] += 1
        d = ORDER[k]; nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if (nr, nc) in cost_map or not master.canMove(d):
            continue
        cost_map[(nr, nc)] = master.move(d)
        if master.isTarget(): target = (nr, nc)
        stack.append([nr, nc, 0])
    return cost_map, target

args = (([[1, 1, 1], [1, 0, 1], [1, 1, 9]], 1, 0, 2, 2))
cm_bad, tg_bad = explore_no_backtrack(GridMaster(*args))
print("bản đồ sai:", sorted(cm_bad.items()), "| đích:", tg_bad)
print("sai:", cheapest_dijkstra(cm_bad, tg_bad), "| đúng:", Solution().findShortestPath(GridMaster(*args)), "| oracle:", oracle(*args))

bản đồ sai: [((-2, -4), 1), ((-2, -3), 1), ((-2, -2), 1), ((-2, -1), 1), ((-2, 0), 1), ((-2, 1), 1), ((-1, -4), 1), ((-1, -3), 1), ((-1, -1), 1), ((-1, 0), 1), ((-1, 1), 1), ((-1, 2), 1), ((0, -4), 1), ((0, -3), 1), ((0, -2), 1), ((0, -1), 1), ((0, 0), 0), ((0, 1), 1), ((0, 2), 1), ((1, -4), 1), ((1, -2), 1), ((1, 0), 1), ((1, 1), 1), ((1, 2), 9), ((2, -4), 1), ((2, -3), 1), ((2, -2), 9), ((3, -4), 1), ((3, -3), 1), ((3, -2), 9)] | đích: (3, -2)
sai: 17 | đúng: 11 | oracle: 11


**Vì sao sai:** sau khi pop ô con, chương trình "nghĩ" robot ở ô cha nhưng robot vẫn ở chỗ cũ ⇒ các `canMove`/`move` sau đó được hỏi ở **sai vị trí**, bản đồ bị lệch/ghi sai tọa độ (có thể xuất hiện ô "ma"). **Sửa:** `master.move(OPP[d])` khi rời ô con.

### Lỗi 2 — Tính cả chi phí ô xuất phát

In [21]:
def wrong_count_start(master):
    cmap, tg = explore(master)
    gm_start_cost = 3                                   # giả sử biết chi phí ô xuất phát (Ví dụ 1)
    r = cheapest_dijkstra(cmap, tg)
    return r + gm_start_cost if r != -1 else -1          # SAI

print("sai:", wrong_count_start(GridMaster(*examples[0][0])), "| đúng:", examples[0][1])

sai: 5 | đúng: 2


**Vì sao sai:** đề nói rõ "chi phí ô xuất phát không bị tính trước khi robot di chuyển". (Thực ra API cũng **không cho biết** chi phí ô xuất phát — ta chỉ biết chi phí khi `move` vào.) **Sửa:** `dist[(0,0)] = 0`.

### Lỗi 3 — Dùng BFS (ít bước nhất) thay vì Dijkstra

In [22]:
def cheapest_bfs_wrong(cost_map, target):
    if target is None: return -1
    q = deque([(0, 0)]); par = {(0, 0): None}
    while q:
        u = q.popleft()
        if u == target: break
        for dr, dc in DIRS.values():
            v = (u[0] + dr, u[1] + dc)
            if v in cost_map and v not in par:
                par[v] = u; q.append(v)
    total, u = 0, target
    while par[u] is not None:
        total += cost_map[u]; u = par[u]
    return total                                        # chi phí của đường ÍT BƯỚC nhất

args = ([[1, 100, 1], [1, 1, 1]], 0, 0, 0, 2)
cm, tg = explore(GridMaster(*args))
print("BFS:", cheapest_bfs_wrong(cm, tg), "| Dijkstra:", cheapest_dijkstra(cm, tg))

BFS: 101 | Dijkstra: 4


**Vì sao sai:** đường 2 bước `→ →` đi qua ô chi phí 100 (tổng 101), còn đường vòng 4 bước rẻ hơn (tổng 4). BFS chỉ đúng khi mọi chi phí bằng nhau (đó là bài anh em LeetCode 1778). **Sửa:** Dijkstra.

### Lỗi 4 — Gọi `move` không kiểm tra, ghi cả giá trị `-1`

In [23]:
def explore_no_check(master, cap=60):
    cost_map = {(0, 0): 0}; target = None
    stack = [[0, 0, 0, None]]
    while stack:
        if len(cost_map) > cap:                           # chặn để demo không chạy vô hạn
            return cost_map, "PHÌNH TO"
        top = stack[-1]; r, c, k, back = top
        if k == 4:
            stack.pop()
            if back: master.move(back)
            continue
        top[2] += 1
        d = ORDER[k]; nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if (nr, nc) in cost_map: continue
        cost_map[(nr, nc)] = master.move(d)              # SAI: có thể là -1 (không di chuyển)
        if master.isTarget(): target = (nr, nc)
        stack.append([nr, nc, 0, OPP[d]])                 # ... và sau đó "quay lui" làm robot đi lạc
    return cost_map, target

args = examples[1][0]
cm_bad, tg_bad = explore_no_check(GridMaster(*args))
print("lưới thật chỉ có", sum(1 for row in args[0] for x in row if x > 0), "ô trống, nhưng bản đồ sai có", len(cm_bad), "ô")
print("số ô 'ma' chi phí -1:", sum(1 for v in cm_bad.values() if v == -1), "| trạng thái:", tg_bad)
print("đúng:", Solution().findShortestPath(GridMaster(*args)))

lưới thật chỉ có 7 ô trống, nhưng bản đồ sai có 61 ô
số ô 'ma' chi phí -1: 59 | trạng thái: PHÌNH TO
đúng: 9


**Vì sao sai:** `move` vào ô chặn trả `-1` và robot **đứng yên**, nhưng chương trình tưởng robot đã sang ô mới (ô "ma" chi phí `-1`) và tiếp tục "khám phá" từ đó — mỗi lần thử ra ngoài lại sinh thêm ô ma, bản đồ **phình ra vô hạn** (ta chặn ở 60 ô để demo dừng). Lệnh "quay lui" sau đó còn làm robot **thật sự** đi sang ô khác. Ngoài ra chi phí `-1` là cạnh âm, phá Dijkstra. **Sửa:** chỉ `move` khi `canMove(d)` (hoặc kiểm tra kết quả `!= -1`).

### Lỗi 5 — DFS khám phá đệ quy trên lưới lớn

In [24]:
def explore_recursive(master):
    cost_map = {(0, 0): 0}; target = [None]
    def dfs(r, c):
        if master.isTarget(): target[0] = (r, c)
        for d in ORDER:
            nr, nc = r + DIRS[d][0], c + DIRS[d][1]
            if (nr, nc) not in cost_map and master.canMove(d):
                cost_map[(nr, nc)] = master.move(d)
                dfs(nr, nc)
                master.move(OPP[d])
    dfs(0, 0)
    return cost_map, target[0]

g = snake(41, 60)
try:
    explore_recursive(GridMaster(g, 0, 0, 40, 59))
    print("không lỗi (giới hạn đệ quy đủ lớn?)")
except RecursionError as e:
    print("RecursionError:", str(e)[:50], "| bản stack:", len(explore(GridMaster(g, 0, 0, 40, 59))[0]), "ô")

RecursionError: maximum recursion depth exceeded | bản stack: 1280 ô


**Vì sao sai:** độ sâu đệ quy bằng độ dài đường DFS (ở lưới rắn ~ một nửa số ô), vượt giới hạn 1000. Có thể `sys.setrecursionlimit` (nhiều lời giải tham khảo làm vậy) nhưng dễ tràn stack C; **sửa an toàn:** stack tường minh như Cách nộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài interactive "robot trong lưới ẩn"** (489 *Robot Room Cleaner*, 1778, 1810) ⇒ DFS khám phá với **tọa độ tương đối** + **quay lui vật lý** bằng hướng ngược.
- Tách **khám phá** (tốn lời gọi API) khỏi **tính toán** (trên dữ liệu đã biết) — đơn giản hóa rất nhiều.
- Chi phí ô bằng nhau ⇒ BFS; chi phí khác nhau, dương ⇒ Dijkstra; nguyên nhỏ ⇒ Dial/0-1 BFS.
- Luôn tự viết **lớp giả lập** + **oracle** để test bài interactive.
- Lưới tới `10⁴` ô ⇒ tránh đệ quy trong Python.

## 15. Biến thể của bài

1. **LeetCode 1778 — Shortest Path in a Hidden Grid:** mọi ô chi phí như nhau, đếm số bước ⇒ khám phá + BFS (Bài tập 4).
2. **LeetCode 489 — Robot Room Cleaner:** chỉ có `move()`/`turnLeft()`/`turnRight()`, phải dọn mọi ô — DFS với quay lui "quay 180°, đi, quay 180°".
3. **Trả về chính đường đi** (chuỗi hướng) đạt chi phí nhỏ nhất — lưu `parent` trong Dijkstra (Bài tập 5).
4. **Giới hạn số lời gọi API** ⇒ dừng khám phá sớm, A* với heuristic.
5. **Chi phí ở cạnh** thay vì ở ô, hoặc có ô "dịch chuyển" — vẫn khám phá + Dijkstra.

## 16. So sánh các cách (pha B; pha A giống nhau)

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn đường đơn | DFS mọi đường, cắt nhánh | hàm mũ | `O(V)` | kiểm chứng lưới tí hon |
| 2. Bellman–Ford | thư giãn lặp tới khi ổn định | `O(V·E)` | `O(V)` | trọng số có thể âm (không ở bài này) |
| 3. Dijkstra | heap, chốt đỉnh nhỏ nhất | `O(E log V)` | `O(V)` | **nộp bài** |
| 4. Dial | xô vòng theo chi phí nguyên ≤ C | `O(V + E + C·V)` | `O(V + C)` | chi phí nguyên nhỏ |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `grid = [[1,1,1]]`, xuất phát `(0,0)`, đích `(0,2)`. Đáp án? Pha A gọi `move` bao nhiêu lần?

**Bài 2 (tính tay):** Với Ví dụ 2, liệt kê thứ tự các ô được Dijkstra **chốt** (dùng tọa độ tương đối ở mục 4) cho tới khi chốt đích.

**Bài 3 (điền TODO):** Viết `api_budget(V, E_tree=None)` trả về **cận trên** số lời gọi `canMove`, `move`, `isTarget` của pha A theo số ô `V` tới được, rồi kiểm bằng giả lập.

**Bài 4 (tự viết lại — LeetCode 1778):** Viết `shortest_steps(master)` trả về **số bước ít nhất** tới đích (hoặc `-1`), tái sử dụng `explore`.

**Bài 5 (mini challenge):** Viết `cheapest_path_dirs(master)` trả về `(chi_phí, chuỗi_hướng)`. Kiểm tra bằng cách **phát lại** chuỗi hướng trên một `GridMaster` mới: mọi `move` phải thành công, tổng chi phí trả về bằng đáp án, và robot kết thúc ở đích.

In [25]:
def api_budget(V):
    # TODO: trả về dict cận trên {"canMove": ..., "move": ..., "isTarget": ...}
    pass

def shortest_steps(master):
    # TODO: explore + BFS
    pass

def cheapest_path_dirs(master):
    # TODO: explore + Dijkstra có parent
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Chỉ có một đường | Đáp án `1 + 1 = 2` | `move`: vào 2 ô + ra 2 lần = 4 |
| 2 | Bắt đầu `(0,0)` d=0 | Kế tiếp `(0,1)` d=2, `(-1,0)` d=3 | … `(-1,1)` d=6, `(-1,2)` d=8, rồi hai ô cùng d=9: `(-2,1)` và đích `(-2,2)` (thứ tự hòa phụ thuộc so sánh tuple) |
| 3 | Mỗi ô thử 4 hướng tối đa | Mỗi ô (trừ gốc) vào 1 lần, ra 1 lần | `isTarget` 1 lần mỗi ô |
| 4 | BFS trên `cost_map` | Mỗi bước +1 | Trả `-1` nếu không có đích |
| 5 | Dijkstra lưu `parent[v] = (u, d)` | Lần ngược từ đích | Đảo chuỗi, phát lại bằng `move` |

## 19. Đáp án tham khảo

In [26]:
gm1 = GridMaster([[1, 1, 1]], 0, 0, 0, 2)
r1 = Solution().findShortestPath(gm1)
print("Bài 1:", r1, "| số move:", gm1.calls["move"]); assert r1 == 2 and gm1.calls["move"] == 4
cm, tg = explore(GridMaster(*examples[1][0]))
print("Bài 2:"); assert cheapest_dijkstra(cm, tg, verbose=True) == 9

Bài 1: 2 | số move: 4
Bài 2:
chốt (0, 0) với chi phí 0
chốt (0, 1) với chi phí 2
chốt (-1, 0) với chi phí 3
chốt (-1, 1) với chi phí 6
chốt (-1, 2) với chi phí 8
chốt (-2, 1) với chi phí 9
chốt (-2, 2) với chi phí 9


In [27]:
def api_budget(V):
    return {"canMove": 4 * V, "move": 2 * (V - 1), "isTarget": V}

def shortest_steps(master):
    cost_map, target = explore(master)
    if target is None:
        return -1
    dist = {(0, 0): 0}; q = deque([(0, 0)])
    while q:
        u = q.popleft()
        if u == target:
            return dist[u]
        for dr, dc in DIRS.values():
            v = (u[0] + dr, u[1] + dc)
            if v in cost_map and v not in dist:
                dist[v] = dist[u] + 1; q.append(v)
    return -1

def oracle_steps(grid, r1, c1, r2, c2):
    ones = [[1 if x > 0 else 0 for x in row] for row in grid]
    return oracle(ones, r1, c1, r2, c2)

rng = random.Random(7)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    if m * n < 2: continue
    g = random_grid(m, n, rng, 0.7)
    (r1, c1), (r2, c2) = rng.sample([(i, j) for i in range(m) for j in range(n)], 2)
    g[r1][c1] = g[r1][c1] or 1; g[r2][c2] = g[r2][c2] or 1
    gm = GridMaster(g, r1, c1, r2, c2)
    cmap, _ = explore(gm)
    b = api_budget(len(cmap))
    assert all(gm.calls[k] <= b[k] for k in b)
    assert shortest_steps(GridMaster(g, r1, c1, r2, c2)) == oracle_steps(g, r1, c1, r2, c2)
print("Bài 3, 4: OK", api_budget(10))

Bài 3, 4: OK {'canMove': 40, 'move': 18, 'isTarget': 10}


In [28]:
def cheapest_path_dirs(master):
    cost_map, target = explore(master)
    if target is None:
        return -1, ""
    dist = {(0, 0): 0}; parent = {(0, 0): None}; pq = [(0, (0, 0))]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        if u == target: break
        for name, (dr, dc) in DIRS.items():
            v = (u[0] + dr, u[1] + dc)
            if v in cost_map and d + cost_map[v] < dist.get(v, float("inf")):
                dist[v] = d + cost_map[v]; parent[v] = (u, name)
                heapq.heappush(pq, (dist[v], v))
    path, u = [], target
    while parent[u] is not None:
        u, name = parent[u]; path.append(name)
    return dist[target], "".join(reversed(path))

rng = random.Random(99)
for _ in range(200):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    if m * n < 2: continue
    g = random_grid(m, n, rng, 0.75)
    (r1, c1), (r2, c2) = rng.sample([(i, j) for i in range(m) for j in range(n)], 2)
    g[r1][c1] = g[r1][c1] or 1; g[r2][c2] = g[r2][c2] or 1
    total, dirs = cheapest_path_dirs(GridMaster(g, r1, c1, r2, c2))
    assert total == oracle(g, r1, c1, r2, c2)
    if total != -1:
        replay = GridMaster(g, r1, c1, r2, c2)
        got = 0
        for ch in dirs:
            step = replay.move(ch); assert step != -1; got += step
        assert got == total and replay.isTarget()
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", cheapest_path_dirs(GridMaster(*examples[1][0])))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK (9, 'RURU')


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Pha A: khám phá (stack), tọa độ tương đối
cost = {(0,0): 0}; target = None; st = [[0, 0, 0, None]]
while st:
    top = st[-1]; r, c, k, ret = top
    if k == 4:
        st.pop()
        if ret: master.move(ret)              # quay lui vật lý
        continue
    top[2] += 1; d = "URDL"[k]; v = (r + dr[d], c + dc[d])
    if v in cost or not master.canMove(d): continue
    cost[v] = master.move(d)
    if master.isTarget(): target = v
    st.append([v[0], v[1], 0, OPP[d]])
# Pha B: Dijkstra từ (0,0) với dist[(0,0)] = 0, cạnh u->v trọng số cost[v]
```

**Checklist:**
- [ ] Chỉ `move` khi `canMove` (không ghi `-1` làm chi phí)?
- [ ] Sau khi khám phá ô con có `move(OPP[d])` để về?
- [ ] Không tính chi phí ô xuất phát (`dist[(0,0)] = 0`)?
- [ ] Dùng Dijkstra (không phải BFS) vì chi phí khác nhau?
- [ ] Không gặp đích ⇒ trả `-1`; khám phá không dùng đệ quy sâu.